# OmniVoice Cocoon Livecommerce V2 - SINGLE Vietnamese Voice

Notebook nay tao TTS tieng Viet cho `data/cocoon_livecommerce_script_v2.json`.

Bat buoc dung **mot voice duy nhat** cho tat ca scene:
- `LANGUAGE_ID = "vi"`
- `VOICE_MODE = "clone"`
- Tat ca scene dung cung mot `REF_AUDIO_PATH`

Reference audio nen dai 3-10 giay, mot nguoi noi tieng Viet, it nhieu, giong phu hop livestream.

In [ ]:
# 1) Install dependencies
# Colab/local Jupyter deu co the chay cell nay.
!pip install -q omnivoice soundfile pandas tqdm

In [ ]:
# 2) Config - DO NOT CHANGE VOICE MODE if you need consistency
from pathlib import Path

LANGUAGE_ID = "vi"          # correct Vietnamese language id
VOICE_MODE = "clone"        # locked: one speaker from one reference audio

# Set this to your one-and-only speaker reference audio.
# Local example: r"data/voices/host_ref.wav"
# Colab example after upload: "/content/host_ref.wav"
REF_AUDIO_PATH = None
REF_TEXT = None              # optional transcript of REF_AUDIO_PATH

INPUT_JSON = Path("data/cocoon_livecommerce_script_v2.json")
OUTPUT_ROOT = Path("data/outputs/omni_tts")

MODEL_ID = "k2-fsa/OmniVoice"
SAMPLE_RATE = 24000
NUM_STEP = 32
SPEED = 1.0
DURATION = None              # keep natural duration; set 5.0 only if you must force length

MAX_SCENES = None            # e.g. 3 for quick test; None = all scenes
START_FROM_SCENE_ID = None   # e.g. "S010"
RESUME = True                # skip existing wav files
SEED = 42

assert LANGUAGE_ID == "vi", "LANGUAGE_ID must stay 'vi' for Vietnamese."
assert VOICE_MODE == "clone", "VOICE_MODE must stay 'clone' to keep one consistent voice."

In [ ]:
# 3) Imports and deterministic seed
import csv
import json
import os
import random
import re
import traceback
import zipfile
from datetime import datetime

import pandas as pd
import soundfile as sf
import torch
from tqdm.auto import tqdm

os.environ.setdefault("PYTHONHASHSEED", str(SEED))
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

In [ ]:
# 4) Load Cocoon livecommerce script JSON
if not INPUT_JSON.exists():
    raise FileNotFoundError(f"Input JSON not found: {INPUT_JSON.resolve()}")

data = json.loads(INPUT_JSON.read_text(encoding="utf-8"))
job_id = data.get("job_id", "cocoon_livecommerce_v2")
scenes = data.get("scenes", [])

if not scenes:
    raise ValueError("JSON does not contain non-empty scenes.")

print("job_id:", job_id)
print("total scenes:", len(scenes))
print("first scene:", scenes[0].get("scene_id"), scenes[0].get("voiceover", "")[:120])

In [ ]:
# 5) Prepare scenes
def safe_filename(name):
    value = str(name or "scene").strip()
    value = re.sub(r"[^a-zA-Z0-9_.-]+", "_", value)
    return value or "scene"

def normalize_text(text):
    value = str(text or "").strip()
    return re.sub(r"\s+", " ", value)

prepared = []
start_enabled = START_FROM_SCENE_ID is None

for scene in scenes:
    scene_id = safe_filename(scene.get("scene_id", "scene"))
    if START_FROM_SCENE_ID and scene_id == START_FROM_SCENE_ID:
        start_enabled = True
    if not start_enabled:
        continue

    text = normalize_text(scene.get("voiceover", ""))
    if not text:
        continue

    prepared.append({
        **scene,
        "scene_id": scene_id,
        "voiceover": text,
        "audio_filename": f"{scene_id}.wav",
        "text_filename": f"{scene_id}.txt",
    })

if MAX_SCENES is not None:
    prepared = prepared[:MAX_SCENES]

if not prepared:
    raise ValueError("No valid scenes with voiceover found.")

pd.DataFrame([
    {
        "scene_id": s["scene_id"],
        "order": s.get("order"),
        "scene_type": s.get("scene_type"),
        "duration_target_sec": s.get("duration_target_sec"),
        "voiceover": s["voiceover"],
    }
    for s in prepared
]).head(10)

In [ ]:
# 6) Set or upload the single reference voice
# This cell intentionally enforces clone mode. All scenes must use the same REF_AUDIO_PATH.
if REF_AUDIO_PATH is None:
    try:
        from google.colab import files
        print("Upload ONE Vietnamese reference audio file, 3-10 seconds, single speaker:")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("Clone mode requires one reference audio file.")
        REF_AUDIO_PATH = next(iter(uploaded.keys()))
    except ImportError:
        raise RuntimeError(
            "Set REF_AUDIO_PATH in Config cell, e.g. r'data/voices/host_ref.wav'. "
            "Clone mode requires one reference audio file."
        )

REF_AUDIO_PATH = str(REF_AUDIO_PATH)
if not Path(REF_AUDIO_PATH).exists():
    raise FileNotFoundError(f"Reference audio not found: {REF_AUDIO_PATH}")

print("LOCKED SINGLE VOICE REF:", REF_AUDIO_PATH)
print("LANGUAGE_ID:", LANGUAGE_ID)

In [ ]:
# 7) Load OmniVoice model
from omnivoice import OmniVoice

if torch.cuda.is_available():
    device_map = "cuda:0"
    dtype = torch.float16
else:
    device_map = "cpu"
    dtype = torch.float32

print("device_map:", device_map)
print("dtype:", dtype)

model = OmniVoice.from_pretrained(
    MODEL_ID,
    device_map=device_map,
    dtype=dtype,
    load_asr=True,
)

In [ ]:
# 8) Generate one wav per scene with the exact same cloned voice
output_dir = OUTPUT_ROOT / safe_filename(job_id)
audio_dir = output_dir / "audio"
text_dir = output_dir / "text"

audio_dir.mkdir(parents=True, exist_ok=True)
text_dir.mkdir(parents=True, exist_ok=True)

voice_profile = {
    "policy": "single_speaker_locked_clone",
    "language_id": LANGUAGE_ID,
    "voice_mode": VOICE_MODE,
    "ref_audio": REF_AUDIO_PATH,
    "ref_text": REF_TEXT,
    "model_id": MODEL_ID,
    "num_step": NUM_STEP,
    "speed": SPEED,
    "duration": DURATION,
    "seed": SEED,
    "created_at": datetime.now().isoformat(timespec="seconds"),
}
(output_dir / "voice_profile.json").write_text(
    json.dumps(voice_profile, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

def call_omnivoice_generate(text):
    kwargs = {
        "text": text,
        "language_id": LANGUAGE_ID,
        "ref_audio": REF_AUDIO_PATH,
        "num_step": NUM_STEP,
        "speed": SPEED,
    }
    if REF_TEXT:
        kwargs["ref_text"] = REF_TEXT
    if DURATION is not None:
        kwargs["duration"] = DURATION

    try:
        return model.generate(**kwargs)
    except TypeError as exc:
        msg = str(exc)
        fallback_keys = ("language_id", "num_step", "speed", "duration")
        if not any(key in msg for key in fallback_keys) and "unexpected keyword" not in msg:
            raise
        print("Fallback: installed OmniVoice does not accept all generation kwargs.")
        minimal_kwargs = {"text": text, "ref_audio": REF_AUDIO_PATH}
        if REF_TEXT:
            minimal_kwargs["ref_text"] = REF_TEXT
        return model.generate(**minimal_kwargs)

def unpack_audio(audio):
    sample_rate = SAMPLE_RATE
    wav = audio
    if isinstance(audio, dict):
        wav = audio.get("audio") or audio.get("wav") or audio.get("waveform")
        sample_rate = int(audio.get("sample_rate") or audio.get("sampling_rate") or sample_rate)
    elif isinstance(audio, tuple):
        wav = audio[0]
        if len(audio) > 1 and isinstance(audio[1], int):
            sample_rate = audio[1]
    elif isinstance(audio, list):
        wav = audio[0]

    if hasattr(wav, "detach"):
        wav = wav.detach().cpu().float().numpy()
    return wav, sample_rate

manifest_rows = []
errors = []

for scene in tqdm(prepared, desc="Generating OmniVoice TTS"):
    scene_id = scene["scene_id"]
    text = scene["voiceover"]
    audio_path = audio_dir / scene["audio_filename"]
    text_path = text_dir / scene["text_filename"]
    text_path.write_text(text, encoding="utf-8")

    if RESUME and audio_path.exists():
        status = "skipped"
        print("SKIP", scene_id, "exists")
    else:
        try:
            audio = call_omnivoice_generate(text)
            wav, sample_rate = unpack_audio(audio)
            sf.write(audio_path, wav, sample_rate)
            status = "ok"
            print("OK", scene_id, "->", audio_path.name)
        except Exception as exc:
            status = "error"
            errors.append({
                "scene_id": scene_id,
                "audio_filename": str(audio_path),
                "error": repr(exc),
                "traceback": traceback.format_exc(),
            })
            print("ERROR", scene_id, repr(exc))

    row = {
        "scene_id": scene_id,
        "clip_id": scene.get("clip_id"),
        "order": scene.get("order"),
        "scene_type": scene.get("scene_type"),
        "duration_target_sec": scene.get("duration_target_sec"),
        "language_id": LANGUAGE_ID,
        "voice_mode": VOICE_MODE,
        "ref_audio": REF_AUDIO_PATH,
        "audio_filename": str(audio_path),
        "text_filename": str(text_path),
        "text": text,
        "status": status,
    }
    manifest_rows.append(row)

manifest_path = output_dir / "tts_manifest.csv"
errors_path = output_dir / "tts_errors.json"

pd.DataFrame(manifest_rows).to_csv(manifest_path, index=False, encoding="utf-8-sig")
errors_path.write_text(json.dumps(errors, ensure_ascii=False, indent=2), encoding="utf-8")

print("Done")
print("OK:", sum(1 for r in manifest_rows if r["status"] == "ok"))
print("Skipped:", sum(1 for r in manifest_rows if r["status"] == "skipped"))
print("Errors:", len(errors))
print("Output dir:", output_dir)
print("Manifest:", manifest_path)

In [ ]:
# 9) Zip output package
zip_path = output_dir.with_suffix(".zip")
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in output_dir.rglob("*"):
        if path.is_file():
            zf.write(path, arcname=str(path.relative_to(output_dir)))

print("ZIP:", zip_path)